# 02 — Simulation Runner

Runs the maritime simulation using the ships generated by `01_ship_generation.ipynb`.

**Prerequisites:**
1. `simulation_config.ipynb` → `simulation_config.json`
2. `01_ship_generation.ipynb` → `simulation_output_data/ships.parquet`

**Outputs** (all in `simulation_output_data/`):
- `edge_statistics.parquet` — traffic per network edge
- `port_occupancy.parquet`  — port utilisation per timestep
- `ship_locations.parquet`  — ship positions per timestep
- `lost_ships.parquet`      — ships that could not complete their voyage
- `checkpoints/`            — periodic state snapshots (resumable)
- `compat/*.csv`            — CSV copies for Part 5 visualisation (if enabled)

---
### Resuming an interrupted run
Set `RESUME_FROM_CHECKPOINT = True` in the cell below to continue from the
most recent checkpoint instead of starting from scratch.

In [9]:
# ============================================================
# Run settings  (not in simulation_config.ipynb)
# ============================================================

# Set to True to resume from the latest checkpoint in simulation_output_data/checkpoints/
RESUME_FROM_CHECKPOINT = True

In [10]:
import json
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import networkx as nx
from tqdm import tqdm

sys.path.insert(0, str(Path('.').resolve()))

from simulation_engine.config_loader import load_config, resolve_paths
from simulation_engine.models import Ship
from simulation_engine.ship_generation import calibrate_port_times
from simulation_engine.simulation_runner import run_simulation
from simulation_engine.io_manager import read_parquet

print('Imports loaded successfully.')

Imports loaded successfully.


In [11]:
# ============================================================
# Load configuration
# ============================================================
cfg = load_config()
cfg = resolve_paths(cfg)

print('Configuration loaded.')
print(f'  Simulation: {cfg["SIMULATION_DAYS"]} days  |  interval: {cfg["INTERVAL_SIZE"]*24:.1f} h')
print(f'  Interruption events: {len(cfg["INTERRUPTION_EVENTS"])}')
print(f'  Economic events:     {len(cfg["ECONOMIC_EVENTS"])}')
print(f'  Checkpoint every:    {cfg["CHECKPOINT_INTERVAL_DAYS"]} days')
print(f'  Backward-compat CSV: {cfg["BACKWARD_COMPAT_CSV"]}')

Configuration loaded.
  Simulation: 365 days  |  interval: 1.0 h
  Interruption events: 0
  Economic events:     0
  Checkpoint every:    30 days
  Backward-compat CSV: True


In [12]:
# ============================================================
# Load network
# ============================================================
print('Loading network...')
with open(cfg['NETWORK_FILE'], 'rb') as f:
    G = pickle.load(f)

print(f'  Nodes: {G.number_of_nodes()}  |  Edges: {G.number_of_edges()}')

Loading network...
  Nodes: 8726  |  Edges: 14634


In [13]:
# ============================================================
# Load ships from ship_generation output
# ============================================================
output_dir = Path(cfg['OUTPUT_DIR'])

ships_parquet = output_dir / 'ships.parquet'
if not ships_parquet.exists():
    raise FileNotFoundError(
        f"ships.parquet not found at {ships_parquet}.\n"
        "Run 01_ship_generation.ipynb first."
    )

print('Loading ships...')
ships_df = pd.read_parquet(ships_parquet)
print(f'  Loaded {len(ships_df):,} ships')

# Load auxiliary lookup data
with open(output_dir / 'common_countries.json') as f:
    common_countries = json.load(f)

with open(output_dir / 'country_to_ports.json') as f:
    country_to_ports = json.load(f)

with open(output_dir / 'port_name_to_node.pkl', 'rb') as f:
    port_name_to_node = pickle.load(f)

print(f'  Countries: {len(common_countries)}')
print(f'  Ports: {sum(len(v) for v in country_to_ports.values())}')

Loading ships...
  Loaded 226,302 ships
  Countries: 168
  Ports: 595


In [14]:
# ============================================================
# Reconstruct Ship objects from ships.parquet
# ============================================================
# ships.parquet stores flat records; Ship dataclasses need full path
# information (a sequence of network node IDs).  Each ship's specific
# origin_port and dest_port are read from ships.parquet (as sampled
# during generation); the matching path is looked up in port_pair_routes.pkl.
# country_pair_optimal.pkl provides a fallback if a specific port pair
# has no pre-computed route.

print('Loading pre-computed routes...')
with open(output_dir / 'port_pair_routes.pkl', 'rb') as f:
    port_pair_routes = pickle.load(f)
with open(output_dir / 'country_pair_optimal.pkl', 'rb') as f:
    country_pair_optimal = pickle.load(f)
print(f'  Port-pair routes:     {len(port_pair_routes):,}')
print(f'  Country-pair optimal: {len(country_pair_optimal):,}')

# Parse HS codes from column names
hs_codes = sorted(
    int(col.replace('cargo_hs', '').replace('_weight', ''))
    for col in ships_df.columns
    if col.startswith('cargo_hs') and col.endswith('_weight')
)

print(f'Reconstructing {len(ships_df):,} Ship objects...')
all_ships = []
skipped   = 0
seed = cfg.get('RANDOM_SEED')
rng  = np.random.default_rng(seed)

for _, row in tqdm(ships_df.iterrows(), total=len(ships_df), desc='Reconstructing ships', unit='ship'):
    o = row['origin_country']
    d = row['dest_country']

    origin_port = row['origin_port']
    dest_port   = row['dest_port']

    # Use per-ship ports from ships.parquet; fall back to country-pair optimal
    # only if the specific port pair has no pre-computed route.
    route_info = port_pair_routes.get((origin_port, dest_port))
    if route_info is None:
        opt = country_pair_optimal.get((o, d))
        if opt is None:
            skipped += 1
            continue
        origin_port = opt['origin_port']
        dest_port   = opt['dest_port']
        route_info  = port_pair_routes.get((origin_port, dest_port))
        if route_info is None:
            skipped += 1
            continue

    cargo_by_hs = {
        hs: {
            'weight': float(row.get(f'cargo_hs{hs}_weight', 0.0)),
            'value':  float(row.get(f'cargo_hs{hs}_value',  0.0)),
        }
        for hs in hs_codes
    }

    ship = Ship(
        id=int(row['ship_id']),
        origin_country=o,
        dest_country=d,
        origin_port=origin_port,
        dest_port=dest_port,
        ship_type=row['ship_type'],
        injection_day=float(row['injection_day']),
        path=route_info['path'],
        path_length=route_info['length'],
        cargo_total_weight=float(row['cargo_total_weight']),
        cargo_total_value=float(row['cargo_total_value']),
        cargo_by_hs=cargo_by_hs,
        loading_time=1,    # recalibrated below
        unloading_time=1,
    )
    all_ships.append(ship)

print(f'  Reconstructed: {len(all_ships):,}  |  Skipped (no route): {skipped}')

# Recalibrate port processing times
calibrate_port_times(
    all_ships,
    port_loading_times=cfg['PORT_LOADING_TIMES'],
    port_unloading_times=cfg['PORT_UNLOADING_TIMES'],
    interval_size_days=cfg['INTERVAL_SIZE'],
    rng=rng,
)

all_ships.sort(key=lambda s: s.injection_day)
print('  Port times calibrated and ships sorted by injection day.')

Loading pre-computed routes...
  Port-pair routes:     345,156
  Country-pair optimal: 28,056
Reconstructing 226,302 Ship objects...


Reconstructing ships: 100%|██████████| 226302/226302 [01:01<00:00, 3693.21ship/s]


  Reconstructed: 226,302  |  Skipped (no route): 0
  Port times calibrated and ships sorted by injection day.


In [15]:
# ============================================================
# Run simulation
# ============================================================
print('Starting simulation...')
print('=' * 60)

run_simulation(
    cfg=cfg,
    G=G,
    all_ships=all_ships,
    common_countries=common_countries,
    country_to_ports=country_to_ports,
    port_name_to_node=port_name_to_node,
    resume_from_checkpoint=RESUME_FROM_CHECKPOINT,
)

Starting simulation...
Canal calibration (target ρ=0.70):
  Suez Canal: 25,627 ships, transit 14h (14 intervals), 59 slot(s)
  Panama Canal: 13,237 ships, transit 22h (22 intervals), 48 slot(s)
No checkpoint found — starting from day 0.
Running simulation: 365 days, 8,760 intervals
  Ships in queue: 226,302


Simulating:   8%|▊         | 721/8760 [02:08<12:36:30,  5.65s/it]

  Checkpoint saved: checkpoint_day_30.0.pkl


Simulating:  16%|█▋        | 1442/8760 [10:11<8:57:30,  4.41s/it] 

  Checkpoint saved: checkpoint_day_60.0.pkl


...

  Checkpoint saved: checkpoint_day_330.0.pkl


Simulating:  99%|█████████▊| 8642/8760 [8:03:08<37:59, 19.32s/it]  

  Checkpoint saved: checkpoint_day_360.0.pkl


Simulating: 100%|██████████| 8760/8760 [8:17:40<00:00,  3.41s/it]



Writing final outputs...
  edge_statistics.parquet  (14,634 edges)
  port_occupancy.parquet   (41,487 records)
  port_cargo.parquet       (579/588 ports with traffic)
  choke_cargo.parquet      (28 choke points)
  lost_ships.parquet       (0 ships lost)

Exporting CSV compatibility files...
  [compat] Wrote simulation_ship_data.csv  (226,302 rows)
  [compat] Wrote simulation_edge_statistics.csv  (14,634 rows)
  [compat] Wrote simulation_port_occupancy.csv  (2,996,666 rows)
  [compat] Wrote simulation_port_cargo.csv  (588 rows)
  [compat] Wrote simulation_choke_cargo.csv  (28 rows)

Completed ships: 216,400
Total weight transported: 12,874,446,277 mt
Total value transported:  $18,900,536,894,738
SIMULATION OUTPUT SUMMARY

Ships: 226,302
  Total weight: 13,481,327,288 mt
  Total value:  $19,624,502,994,773
  Rerouted:     0

Lost ships: 0

Edges:  14,634 total, 11,666 with traffic

Ports:  588 total, 579 with traffic

Choke points: 28
  Korea Strait: 19,385 ships, 1,145,515,050 mt, $2,3

In [16]:
# ============================================================
# Quick verification
# ============================================================
print('\nOutput files:')
for fname in [
    'ships.parquet', 'edge_statistics.parquet',
    'port_occupancy.parquet', 'ship_locations.parquet', 'lost_ships.parquet',
    'port_cargo.parquet', 'choke_cargo.parquet',
]:
    p = output_dir / fname
    if p.exists():
        df = pd.read_parquet(p)
        size_kb = p.stat().st_size / 1024
        print(f'  {fname:<35} {len(df):>8,} rows  {size_kb:>8.0f} KB')
    else:
        print(f'  {fname:<35} NOT FOUND')

if (output_dir / 'compat').exists():
    print('\nCompat CSV files:')
    for f in sorted((output_dir / 'compat').glob('*.csv')):
        size_kb = f.stat().st_size / 1024
        print(f'  {f.name:<40} {size_kb:>8.0f} KB')

print('\nNext step: run part_5_visualization notebooks')


Output files:
  ships.parquet                        226,302 rows     47559 KB
  edge_statistics.parquet               14,634 rows      5892 KB
  port_occupancy.parquet              2,996,666 rows      3356 KB
  ship_locations.parquet              83,018,883 rows   1053323 KB
  lost_ships.parquet                         0 rows         5 KB
  port_cargo.parquet                       588 rows       850 KB
  choke_cargo.parquet                       28 rows       167 KB

Compat CSV files:
  simulation_choke_cargo.csv                     99 KB
  simulation_edge_statistics.csv              30258 KB
  simulation_port_cargo.csv                    1615 KB
  simulation_port_occupancy.csv              102142 KB
  simulation_ship_data.csv                   247722 KB

Next step: run part_5_visualization notebooks
